# Simulator — the shared engine

This notebook holds **everything all approaches share**: settings, data, devices, model, clients, the event loop, and saving results.

**How it's used:** every approach notebook starts with

```python
%run ../simulator.ipynb
```

That runs this whole notebook and gives the approach every function and class defined below. Running it on its own does nothing visible; it only defines things.

> **Important:** a change here affects **every** approach. To try a new idea, write a new notebook in `approaches/` instead of editing this one.

| Section | What it does |
|---|---|
| 1. Settings | reads `settings.yaml` |
| 2. Data | loads MNIST, splits it across clients (IID or non-IID) |
| 3. Devices | speed and latency of each client (heterogeneity) |
| 4. Model | the neural network |
| 5. Client | real PyTorch training, cut into virtual-time slices |
| **6. Approach API ** | **the only section you need to read to write a new approach** |
| 7. Helpers | `fedavg`, `apply_delta` |
| 8. Server | the event-driven loop |
| 9. `run()` | runs an approach for every seed and saves its results |

## 1. Settings

Reads `settings.yaml` (found automatically in this folder or a parent folder) into `S`, so you can write `S.num_clients`, `S.lr`, and so on.

> **Note:** `settings_id(S)` is a short fingerprint of the settings. Every result is tagged with it, so `compare.ipynb` only compares results produced under the **same** settings.

In [ ]:
import copy, hashlib, heapq, itertools, json, random, time
from dataclasses import dataclass
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import yaml

# the project folder = the first folder (here or above) that contains settings.yaml
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "settings.yaml").exists())
RESULTS = ROOT / "results"


# optional settings: if a settings file doesn't mention them, these values are used
DEFAULTS = {
    "validation_samples": None,   # held-out training images used to score tuning (no client has them)
    "speed_batch_size": None,     # if set, speeds are batches of this size, so bigger batches take longer
    "eval_interval": None,        # if set, evaluate every N virtual seconds instead of every eval_every versions
}


def load_settings(file="settings.yaml", **overrides):
    with open(ROOT / file) as f:
        values = {**DEFAULTS, **yaml.safe_load(f)}
    for key, value in overrides.items():
        if key not in values:
            raise KeyError(f"unknown setting '{key}'")
        values[key] = value
    return SimpleNamespace(**values)


def settings_id(S):
    """Fingerprint of the settings that change results (seeds/paths/device excluded)."""
    ignore = {"seeds", "data_dir", "device"}
    d = {k: v for k, v in sorted(vars(S).items()) if k not in ignore}
    return hashlib.md5(json.dumps(d, default=str).encode()).hexdigest()[:8]


S = load_settings()

## 2. Data

* The dataset is loaded **once** into memory as tensors. That's much faster than transforming images one at a time, and gives exactly the same values.
* `partition: iid` gives every client a random, equal share.
* `partition: dirichlet` makes clients **non-IID**: each one gets a different mix of digits. A smaller `dirichlet_alpha` means more skew.

> **Note:** `label_mix` records each client's share of each digit. The server uses it to score the global model **on every client's own data mix**; that is the fairness metric (`worst10_client_acc`).

In [ ]:
DATASETS = {   # torchvision name, per-channel mean, per-channel std
    "mnist": ("MNIST", (0.1307,), (0.3081,)),
    "fashion_mnist": ("FashionMNIST", (0.2860,), (0.3530,)),
    "cifar10": ("CIFAR10", (0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
}


def load_data(S, seed):
    from torchvision import datasets
    name, mean, std = DATASETS[S.dataset]
    root = str((ROOT / S.data_dir).resolve())

    def raw(train):
        ds = getattr(datasets, name)(root, train=train, download=True)
        x = torch.as_tensor(np.asarray(ds.data))
        x = x.unsqueeze(1) if x.ndim == 3 else x.permute(0, 3, 1, 2)      # -> [N, channels, H, W]
        return x, torch.as_tensor(np.asarray(ds.targets), dtype=torch.long)

    def normalise(x):
        # same values as ToTensor() + Normalize()
        x = x.to(torch.float32).div(255)
        return x.sub(torch.tensor(mean)[:, None, None]).div(torch.tensor(std)[:, None, None]).contiguous()

    train_raw, train_y = raw(True)
    test_raw, test_y = raw(False)
    rng = np.random.default_rng(seed)
    train_idx = rng.choice(len(train_y), size=S.train_samples or len(train_y), replace=False)
    test_idx = rng.choice(len(test_y), size=S.test_samples or len(test_y), replace=False)

    if S.partition == "iid":
        shuffled = np.array(train_idx).copy()
        np.random.default_rng(seed).shuffle(shuffled)
        parts = [c.tolist() for c in np.array_split(shuffled, S.num_clients)]
    else:
        parts = dirichlet_split(train_y.numpy(), train_idx, S.num_clients, S.dirichlet_alpha, seed)

    label_mix = np.array([np.bincount(train_y.numpy()[p], minlength=10) for p in parts], dtype=float)
    label_mix /= label_mix.sum(1, keepdims=True)

    # validation set: training images that no client holds (used to tune without touching the test set)
    val_x = val_y = None
    if S.validation_samples:
        unused = np.setdiff1d(np.arange(len(train_y)), train_idx)
        if len(unused) < S.validation_samples:
            raise ValueError("validation_samples needs training images left over: lower train_samples")
        val_idx = torch.as_tensor(np.random.default_rng([seed, 99]).choice(unused, S.validation_samples, replace=False))
        val_x, val_y = normalise(train_raw[val_idx]), train_y[val_idx]

    # keep only the training images clients use (saves memory) and renumber the client indices
    keep = np.asarray(train_idx)
    new_index = np.full(len(train_y), -1)
    new_index[keep] = np.arange(len(keep))
    parts = [new_index[np.asarray(p, dtype=int)].tolist() for p in parts]
    keep_t, test_t = torch.as_tensor(keep), torch.as_tensor(test_idx)
    return SimpleNamespace(train_x=normalise(train_raw[keep_t]), train_y=train_y[keep_t],
                           test_x=normalise(test_raw[test_t]), test_y=test_y[test_t],
                           val_x=val_x, val_y=val_y, parts=parts, label_mix=label_mix,
                           shape=tuple(train_raw.shape[1:]))


def dirichlet_split(labels, selected, num_clients, alpha, seed):
    """For every digit, split its samples across clients with Dirichlet(alpha) proportions."""
    rng = np.random.default_rng(seed)
    sel_labels = labels[selected]
    for _ in range(100):
        result = [[] for _ in range(num_clients)]
        for cls in np.unique(sel_labels):
            idx = np.asarray(selected)[sel_labels == cls].copy()
            rng.shuffle(idx)
            p = rng.dirichlet(np.full(num_clients, alpha))
            for i, chunk in enumerate(np.split(idx, (np.cumsum(p)[:-1] * len(idx)).astype(int))):
                result[i].extend(chunk.tolist())
        if all(result):
            for group in result:
                rng.shuffle(group)
            return result
    raise RuntimeError("Could not give every client data; increase dirichlet_alpha.")

## 3. Devices (heterogeneity)

Each client has:
* **speed**: how many batches it trains per virtual second
* **download / upload latency**: virtual seconds to receive and to send a model

These come from the lists in `settings.yaml`. If a list is set to `random`, values are drawn from a long-tailed distribution (`random_spread` controls how different devices are).

> **Note:** slow speed plus long latency is what makes a **straggler**.

In [ ]:
def make_devices(S, seed):
    rng = np.random.default_rng([seed, 7])
    n = S.num_clients

    def pick(value, median):
        if value == "random":
            return rng.lognormal(np.log(median), S.random_spread, n).tolist()
        if len(value) != n:
            raise ValueError(f"settings.yaml: device lists need {n} values (num_clients), got {len(value)}")
        return value

    speeds = [max(s, 0.05) for s in pick(S.speeds, 1.0)]
    return speeds, pick(S.download_latency, 1.0), pick(S.upload_latency, 2.0)

## 4. Model

`simple_nn` is the MLP from the original notebooks (784 → 128 → 64 → 10). `cnn` is a small convolutional net. Pick one with `model:` in `settings.yaml`.

In [ ]:
class SimpleNN(nn.Module):
    """The original notebooks' MLP (input -> 128 -> 64 -> 10)."""
    def __init__(self, shape):
        super().__init__()
        self.fc1, self.fc2, self.fc3 = nn.Linear(int(np.prod(shape)), 128), nn.Linear(128, 64), nn.Linear(64, 10)

    def forward(self, x):
        x = x.reshape(x.size(0), -1)
        return self.fc3(F.relu(self.fc2(F.relu(self.fc1(x)))))


class CNN(nn.Module):
    """Two conv layers + two dense layers (for 28x28 grayscale)."""
    def __init__(self, shape):
        super().__init__()
        c, side = shape[0], shape[-1]
        self.c1, self.c2 = nn.Conv2d(c, 32, 5, padding=2), nn.Conv2d(32, 64, 5, padding=2)
        self.f1, self.f2 = nn.Linear(64 * (side // 4) ** 2, 512), nn.Linear(512, 10)

    def forward(self, x):
        x = F.max_pool2d(F.relu(self.c1(x)), 2)
        x = F.max_pool2d(F.relu(self.c2(x)), 2)
        return self.f2(F.relu(self.f1(x.flatten(1))))


class CifarCNN(nn.Module):
    """Three 3x3 conv blocks (32-64-64 channels) + 128-unit dense layer, ~190k parameters.
    Sized to train at a practical speed on a laptop CPU."""
    def __init__(self, shape):
        super().__init__()
        c, side = shape[0], shape[-1]
        self.c1, self.c2, self.c3 = nn.Conv2d(c, 32, 3, padding=1), nn.Conv2d(32, 64, 3, padding=1), nn.Conv2d(64, 64, 3, padding=1)
        self.f1, self.f2 = nn.Linear(64 * (side // 8) ** 2, 128), nn.Linear(128, 10)

    def forward(self, x):
        for conv in (self.c1, self.c2, self.c3):
            x = F.max_pool2d(F.relu(conv(x)), 2)
        return self.f2(F.relu(self.f1(x.flatten(1))))


MODELS = {"simple_nn": SimpleNN, "cnn": CNN, "cifar_cnn": CifarCNN}


def build_model(S, shape):
    return MODELS[S.model](shape)

## 5. Client

A client **really trains** with PyTorch, but its training is spread over **virtual time**:

1. Each time slice (`time_quantum`), the client gains `speed × time_quantum` work credit.
2. Every full credit trains **one batch**; leftover fractions carry over to the next slice.
3. After `local_epochs` epochs it is done, and its update reaches the server `upload_latency` seconds later.

> **Important:** training finishes at the **end** of the slice where the last batch ran. This is the same convention as the original notebooks.

When a client finishes, it produces an **`Update`**. That's what your approach receives in `on_update`.

In [ ]:
@dataclass
class Update:
    """What reaches the server when a client finishes."""
    client_id: int
    state: dict             # the trained model weights
    start_state: dict       # the weights the client started from
    samples: int            # client's dataset size
    trained_version: int    # global version the client trained on
    staleness: int          # how many new global versions appeared while it trained
    arrival_time: float
    download_time: float
    batches: int
    train_loss: float

    @property
    def delta(self):
        """trained weights - start weights (what the client learned)."""
        if not hasattr(self, "_delta"):
            self._delta = {k: (v - self.start_state[k]) if torch.is_floating_point(v) else torch.zeros_like(v)
                           for k, v in self.state.items()}
        return self._delta


class Client:
    def __init__(self, cid, indices, speed, download, upload, data, S, shuffle_rng, device):
        self.id, self.indices, self.samples = cid, list(indices), len(indices)
        self.speed, self.download_latency, self.upload_latency = speed, download, upload
        self.data, self.S, self.rng, self.device = data, S, shuffle_rng, device
        self.busy = False
        self.model = None

    def begin(self, weights, version, local_epochs, lr, extra_loss):
        S = self.S
        self.version, self.local_epochs, self.extra_loss = version, local_epochs, extra_loss
        self.model = build_model(S, self.data.shape).to(self.device)
        self.model.load_state_dict(weights)
        self.start_state = {k: v.detach().cpu().clone() for k, v in weights.items()}
        self.optimizer = torch.optim.SGD(self.model.parameters(), lr=lr, momentum=S.momentum)
        self.epoch = self.batches = 0
        self.credit = self.loss_sum = 0.0
        self.finished = False
        self.finish_time = self.arrival_time = None
        self._shuffle()

    def _shuffle(self):
        self.order = self.indices.copy()
        self.rng.shuffle(self.order)
        self.cursor = 0

    def _train_batch(self):
        ids = torch.as_tensor(self.order[self.cursor:self.cursor + self.S.batch_size], dtype=torch.long)
        x, y = self.data.train_x[ids].to(self.device), self.data.train_y[ids].to(self.device)
        self.model.train()
        self.optimizer.zero_grad()
        loss = F.cross_entropy(self.model(x), y)
        extra = self.extra_loss(self, self.model)
        (loss if extra is None else loss + extra).backward()
        self.optimizer.step()
        self.loss_sum += loss.item()
        self.batches += 1
        self.cursor += len(ids)
        if self.cursor >= len(self.order):
            self.epoch += 1
            if self.epoch >= self.local_epochs:
                return True
            self._shuffle()
        return False

    def train_for(self, start, quantum):
        """Train for one virtual time slice."""
        if self.finished or start < self.ready_time:
            return 0
        # cost of one batch in "speed units": 1 by default; batch_size / speed_batch_size when that is set
        cost = self.S.batch_size / self.S.speed_batch_size if self.S.speed_batch_size else 1.0
        self.credit += self.speed * quantum
        n = 0
        while self.credit + 1e-9 >= cost and not self.finished:
            done = self._train_batch()
            self.credit -= cost
            n += 1
            if done:
                self.finished = True
                self.finish_time = start + quantum              # finishes at the END of the slice
                self.arrival_time = self.finish_time + self.upload_latency
        return n

    def make_update(self, server_version, now):
        u = Update(self.id, {k: v.detach().cpu().clone() for k, v in self.model.state_dict().items()},
                   self.start_state, self.samples, self.version, server_version - self.version, now,
                   self.download_time, self.batches, self.loss_sum / max(self.batches, 1))
        self.model = self.optimizer = self.start_state = None   # free memory until the next job
        return u

## 6. Approach API

**This is the only section you need to read to write a new approach.** An approach is a class that subclasses `Approach` and overrides what it needs:

| Method | Called when | Default | Override it for |
|---|---|---|---|
| `start(server)` | t = 0 | every client starts training | client selection |
| `client_free(server, client)` | a client delivered its update | it trains again immediately | sync rounds, tiers, waiting |
| **`on_update(server, update)`** | **an update arrives** | **must be written** | **aggregation, staleness, buffering** |
| `local_loss(client, model)` | every local batch | none | FedProx-style terms (non-IID) |
| `local_epochs(server, client)` | a client starts a job | `local_epochs` from settings | less work for slow devices |

**What you can use on `server`:**

| | |
|---|---|
| `server.model` | current global weights |
| `server.version` | number of global updates so far |
| `server.time` | current virtual time |
| `server.clients` | list of clients (`.id`, `.samples`, `.speed`, `.download_latency`, `.upload_latency`, `.busy`) |
| `server.rng` | random generator (e.g. for picking clients) |
| `server.train(client_id, at=None)` | start a client now, or at virtual time `at` |
| `server.set_model(weights, updates)` | publish a new global model |

Give each approach a `label`: it is the name shown in plot legends.

**What an `update` has:** `.client_id` `.state` `.delta` `.samples` `.staleness` `.trained_version` `.arrival_time` `.batches` `.train_loss`

> **Tip:** approach parameters are class attributes (e.g. `buffer_size = 5`). To try a variant, write `FedBuff(buffer_size=10)`. It is saved as a separate run named `fedbuff_buffer_size=10`.

In [ ]:
class Approach:
    name = None                    # results are saved under this name (default: class name in lowercase)
    label = None                   # name shown in plot legends (default: the run name)
    copy_model_at = "download"     # when clients copy the global model: "download" or "dispatch"

    def __init__(self, **params):
        for key, value in params.items():
            if not hasattr(self, key):
                raise AttributeError(f"{type(self).__name__} has no parameter '{key}'")
            setattr(self, key, value)
        self.changed = params

    def run_name(self):
        base = self.name or type(self).__name__.lower()
        return base + "".join(f"_{k}={v}" for k, v in self.changed.items())

    def display_label(self):
        base = self.label or self.run_name()
        if self.label and self.changed:
            base += " (" + ", ".join(f"{k}={v}" for k, v in self.changed.items()) + ")"
        return base

    # ---------------- override these ----------------
    def start(self, server):
        for c in server.clients:
            server.train(c.id)

    def client_free(self, server, client):
        server.train(client.id)

    def on_update(self, server, update):
        raise NotImplementedError("on_update must be written")

    def local_loss(self, client, model):
        return None

    def local_epochs(self, server, client):
        return server.S.local_epochs

## 7. Helpers for approaches

* `fedavg(updates)`: the weighted average of the clients' **full models**, by sample count unless you pass `weights`.
* `apply_delta(model, updates, weights, rate)`: `model + rate × weighted average of the updates' deltas`. Each update is applied on top of the **current** global model.

> **Important:** `apply_delta` normalises `weights` so they sum to 1. With a **single** update, any weight cancels out (w / w = 1). To scale one update by staleness, change **`rate`**, not the weight. This is exactly why the original notebook 03 behaved like notebook 02.

In [ ]:
def fedavg(updates, weights=None):
    states = [u.state for u in updates]
    weights = weights or [u.samples for u in updates]
    total = sum(weights)
    return {k: sum(states[i][k] * (weights[i] / total) for i in range(len(states))) for k in states[0]}


def apply_delta(model, updates, weights, rate):
    total = sum(weights)
    out = {}
    for k, cur in model.items():
        if not torch.is_floating_point(cur):
            out[k] = cur.clone()
            continue
        out[k] = cur + rate * sum((u.delta[k] * (w / total) for u, w in zip(updates, weights)), torch.zeros_like(cur))
    return out

## 8. Server: the event-driven loop

Every client job is a chain of events, processed in virtual-time order:

```
START ─► DOWNLOAD_DONE ─► SLICE_END ─► SLICE_END ─► … ─► UPDATE_ARRIVED
          (+download)      (batches really train here)    (+upload → approach.on_update)
```

* Events at the **same** time are ordered by a fixed priority, so every run is deterministic.
* After every `set_model`, the global model is evaluated (every `eval_every` versions), and accuracy, loss, throughput, staleness and fairness are recorded.
* The run stops at `max_updates` updates or `max_time` virtual seconds.

> **Note:** when the update budget is used up, no new client jobs are started, so no work is wasted on updates that would never count.

In [ ]:
PRIORITY = {"DOWNLOAD_DONE": 0, "UPDATE_ARRIVED": 1, "SLICE_END": 2, "START": 3}


class Server:
    def __init__(self, S, approach, data, clients, net, device, seed):
        self.S, self.approach, self.data, self.clients, self.net, self.device = S, approach, data, clients, net, device
        self.rng = np.random.default_rng([seed, 3])
        self.model = copy.deepcopy(net.state_dict())
        self.version, self.time = 0, 0.0
        self.jobs = self.received = 0
        self.queue, self.counter = [], itertools.count()
        self.metrics, self.update_log, self.pending = [], [], []
        self.wall0 = time.perf_counter()
        self.next_eval_time = S.eval_interval or 0.0

    # ---------- API for approaches ----------
    def train(self, client_id, at=None):
        if at is not None and at > self.time:
            self._push(at, "START", client_id)
            return True
        c = self.clients[client_id]
        if c.busy:
            raise RuntimeError(f"client {client_id} is already training")
        if self.S.max_updates is not None and self.jobs >= self.S.max_updates:
            return False
        self.jobs += 1
        c.busy = True
        if self.approach.copy_model_at == "dispatch":
            self._begin(c)
        self._push(self.time + c.download_latency, "DOWNLOAD_DONE", client_id)
        return True

    def set_model(self, weights, updates):
        self.model = weights
        self.net.load_state_dict(weights)
        self.version += 1
        self.pending.extend(updates)
        if self.S.eval_interval:                         # evaluate every eval_interval virtual seconds
            if self.time >= self.next_eval_time:
                self._evaluate()
                self.next_eval_time = (self.time // self.S.eval_interval + 1) * self.S.eval_interval
        elif self.version % self.S.eval_every == 0:
            self._evaluate()

    # ---------- internals ----------
    def _push(self, when, kind, cid):
        heapq.heappush(self.queue, (float(when), PRIORITY[kind], next(self.counter), kind, cid))

    def _begin(self, c):
        c.begin(self.model, self.version, self.approach.local_epochs(self, c), self.S.lr, self.approach.local_loss)

    @torch.no_grad()
    def _evaluate(self):
        self.net.eval()
        loss = correct = 0.0
        hit_per_class, count_per_class = torch.zeros(10), torch.zeros(10)
        for s in range(0, len(self.data.test_y), 256):
            x, y = self.data.test_x[s:s + 256].to(self.device), self.data.test_y[s:s + 256].to(self.device)
            out = self.net(x)
            loss += F.cross_entropy(out, y, reduction="sum").item()
            hit = out.argmax(1) == y
            correct += hit.sum().item()
            hit_per_class += torch.bincount(y[hit].cpu(), minlength=10)
            count_per_class += torch.bincount(y.cpu(), minlength=10)
        n = len(self.data.test_y)
        client_acc = self.data.label_mix @ (hit_per_class / count_per_class.clamp(min=1)).double().numpy()
        st = [u.staleness for u in self.pending]
        self.metrics.append(dict(
            version=self.version, time=self.time, updates=self.received, accuracy=correct / n, loss=loss / n,
            throughput=self.received / self.time if self.time > 0 else 0.0,
            mean_staleness=float(np.mean(st)) if st else 0.0, max_staleness=max(st) if st else 0,
            worst10_client_acc=float(np.percentile(client_acc, 10)), min_client_acc=float(client_acc.min()),
            wall_seconds=time.perf_counter() - self.wall0))
        if self.data.val_x is not None:                  # validation score (used for tuning)
            vloss = vcorrect = 0.0
            for s in range(0, len(self.data.val_y), 256):
                x, y = self.data.val_x[s:s + 256].to(self.device), self.data.val_y[s:s + 256].to(self.device)
                out = self.net(x)
                vloss += F.cross_entropy(out, y, reduction="sum").item()
                vcorrect += (out.argmax(1) == y).sum().item()
            self.metrics[-1].update(val_accuracy=vcorrect / len(self.data.val_y), val_loss=vloss / len(self.data.val_y))
        self.pending = []

    def run(self):
        S, q = self.S, self.S.time_quantum
        self._evaluate()                                   # the starting model, at t = 0
        self.approach.start(self)
        while self.queue:
            if S.max_updates is not None and self.received >= S.max_updates:
                break
            if S.max_time is not None and self.queue[0][0] > S.max_time:
                break
            when, _, _, kind, cid = heapq.heappop(self.queue)
            self.time, c = when, self.clients[cid]
            if kind == "START":
                self.train(cid)
            elif kind == "DOWNLOAD_DONE":
                if self.approach.copy_model_at == "download":
                    self._begin(c)
                c.download_time = c.ready_time = when
                self._push(when + q, "SLICE_END", cid)
            elif kind == "SLICE_END":
                c.train_for(when - q, q)
                if c.finished:
                    self._push(c.arrival_time, "UPDATE_ARRIVED", cid)
                else:
                    self._push(when + q, "SLICE_END", cid)
            elif kind == "UPDATE_ARRIVED":
                self.received += 1
                u = c.make_update(self.version, when)
                c.busy = False
                self.approach.on_update(self, u)
                self.update_log.append(dict(update=self.received, client=u.client_id, staleness=u.staleness,
                                            trained_version=u.trained_version, samples=u.samples,
                                            download_time=u.download_time, arrival_time=u.arrival_time,
                                            batches=u.batches, train_loss=u.train_loss))
                self.approach.client_free(self, c)
        if self.metrics[-1]["version"] != self.version:
            self._evaluate()                               # always score the final model

## 9. `run(approach)`

Runs the approach once per seed in `settings.yaml` (or the `seeds` you pass) and saves two files per seed:

* `results/<name>_seed<k>_metrics.csv`: one row per evaluation (accuracy, loss, time, throughput, staleness, fairness)
* `results/<name>_seed<k>_updates.csv`: one row per client update (who, how stale, when)

Running the same approach and seed again **overwrites** its own files only.

> **Tip:** `run(...)` returns the metrics table, so the approach notebook can show it right away.

In [ ]:
def run(approach, seeds=None, verbose=True, settings_file="settings.yaml", overrides=None, results_dir=None, name=None):
    """overrides: change settings for this run only, e.g. {"lr": 0.05}. results_dir: where to save (default results/)."""
    S = load_settings(settings_file, **(overrides or {}))   # re-read, in case settings.yaml changed
    out_dir = Path(results_dir) if results_dir else RESULTS
    name, label, sid = name or approach.run_name(), approach.display_label(), settings_id(S)
    tables = []
    for seed in seeds or S.seeds:
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        device = torch.device(("cuda" if torch.cuda.is_available() else "cpu") if S.device == "auto" else S.device)
        data = load_data(S, seed)
        net = build_model(S, data.shape).to(device)          # first random draw after seeding (as the notebooks)
        speeds, down, up = make_devices(S, seed)
        clients = [Client(i, data.parts[i], speeds[i], down[i], up[i], data, S,
                          random if S.shared_shuffle_rng else random.Random(seed * 1_000_003 + i), device)
                   for i in range(S.num_clients)]
        a = copy.deepcopy(approach)                         # fresh approach state for every seed
        server = Server(S, a, data, clients, net, device, seed)
        server.run()

        out_dir.mkdir(parents=True, exist_ok=True)
        m = pd.DataFrame(server.metrics).assign(run=name, label=label, seed=seed, settings_id=sid, num_clients=S.num_clients)
        u = pd.DataFrame(server.update_log).assign(run=name, seed=seed, settings_id=sid)
        m.to_csv(out_dir / f"{name}_seed{seed}_metrics.csv", index=False)
        u.to_csv(out_dir / f"{name}_seed{seed}_updates.csv", index=False)
        tables.append(m)
        if verbose:
            last = m.iloc[-1]
            val = f"val={last.val_accuracy * 100:6.2f}%  " if "val_accuracy" in m else ""
            print(f"{name:28s} seed={seed:<4d} {val}acc={last.accuracy * 100:6.2f}%  time={last.time:8.1f}s  "
                  f"updates={int(last.updates):5d}  worst10%={last.worst10_client_acc * 100:6.2f}%  "
                  f"wall={last.wall_seconds:.1f}s")
    return pd.concat(tables, ignore_index=True)


def load_approach(name):
    """Return the Approach class defined in approaches/<name>.ipynb, without running that notebook's experiment."""
    import nbformat
    nb = nbformat.read(ROOT / "approaches" / f"{name}.ipynb", as_version=4)
    namespace = dict(globals())
    before = dict(namespace)
    for cell in nb.cells:
        if cell.cell_type == "code" and "(Approach)" in cell.source:
            exec(cell.source, namespace)
    found = [v for k, v in namespace.items() if before.get(k) is not v
             and isinstance(v, type) and issubclass(v, Approach) and v is not Approach]
    if len(found) != 1:
        raise ValueError(f"expected one Approach class in approaches/{name}.ipynb, found {len(found)}")
    return found[0]